In [ ]:
import numpy as np
import scipy
from scipy.io.wavfile import read
import matplotlib.pyplot as plt
from IPython.display import Audio
import librosa, librosa.display
plt.rcParams['figure.figsize'] = (14, 5)

# Onset Detection

Onset detection is a common MIR task where we attempt to segment the audio into musical events. It helps us perform segmentation, tempo estimation, meter estimation, source separation, transcription, and more.

There are multiple ways to perform onset detection. Generally, we follow these steps:
1. Break down the signal into relevant feature(s)
2. Apply a detection function (also called novelty function)
3. Employ a peak picking or thresholding algorithm to identify *relevant* onsets

### Feature based Onset Detection

There are many strategies and techniques for onset detection using different initial feature sets to re-represent the audio data. Here are a few outlined in chapter 16 of *Music Data Analysis* by Weihs et al:
- time-domain: ZCR, energy/amplitude envelope
- spectral: STFT with energy envelope, spectral flux, spectral centroid, MFCCs
- "rhythmic": high frequency content, phase deviation, and complex domain analysis

Today, we'll focus on energy-based detection.

## Some Definitions

**Attack** : sharp increase of energy

**Transient** : a short duration with high amplitude
within which signal evolves quickly

**Onset** : single instant marking the beginning of
transient. (Onsets frequently occur on beats.)

## Energy-based Onset Detection

Here is a general outline for energy-based onset detection:

* Compute RMS (you may also try Energy or log Energy)
* Compute Derivative (differentiation) 
$\Delta E(n) = E(n+1) - E(n) $ 
* Perform half wave rectification (above or equal to zero; only energy increases are relevant for note onsets).
* Use a thresholding procedure where energy above some cutoff equals "1" (on) to identify onsets.

Before implementing this algorithm, let's revisit energy and RMSE.

#### Analogy: Energy as "loudness" and/or "effort"

Think of energy like how much effort it takes to play a musical note. If you bang a drum hard, that’s high energy (and louder); if you tap it lightly, that’s low energy (and quieter).

The formula for the total **energy** of a discrete signal (or window) is defined by:

$$ \sum_n \left| x(n) \right|^2 $$

If we apply this over every $n$ (sample) in our discrete time signal, this corresponds to the *total* (sum) magntiude of the signal. This roughly corresponds to how loud a signal is. Note that we square the values before summing: this does two things:

* it ensures that we get rid of negative values (which would otherwise cancel out with the positive)
* it better aligns with human loudness perception.

#### RMS energy: the average magnitude of a signal

With RMSE, we have an averaging, we account for the number of values being summed by including a $\frac{1}{N}$ normalization factor prior to taking the square root. (I.e., we calculate the square *root* of the *mean* of the *squared* values!).

The **root-mean-square energy (RMSE)** in a signal is defined as

$$ \sqrt{ \frac{1}{N} \sum_n \left| x(n) \right|^2 } $$

* We average the squared values first.

* Then we take the square root to bring it back to the same unit as the signal amplitude.

* RMS is useful because it gives a consistent notion of signal strength — it’s more comparable across signals.

Notice that we could also calculate the energy after we have transformed to spectral energy. That is, the energy of a given time frame must be spread in some way across the frequency band (this is what we are measuring when we are measuring the magnitude spectrum). As such, if we simply summed over all the bands, we would know the total magnitude for that time frame. 


(Note, if you are *already* calculating other spectral features via a STFT procedure, you may wish to compute the energy or RMSE that way. However, for today, we will calculate energy directly from the discrete time signal.

### Clarifications: Magnitude vs. Amplitude vs. Energy

**Magnitude** typically refers to the size or strength of the signal *without any regard for direction* (polarity). In other words when we talk about the "magnitude of a signal" we are referring to values in absolute size. RMS is often described as the "average magnitude"

**Amplitude** can refer to either:
* the instantaneous value of the signal (e.g. voltage) at any given time, and
* the **maximum** value of the signal (peak amplitude)
More formally in *this* context, amplitude is the latter; the absolute value of the signal's peak at any point in time.

**Energy** refers to the "total amount" of signal strength over time.

Thus RMS is a time-averaged magnitude, while energy is the total accumulated magnitude over a period of time.



Let's load a signal:

In [ ]:
#x, sr = librosa.load('../audio/CongaGroove-mono.wav')
(fs, x) = read('../audio/CongaGroove-mono.wav')

Listen to the signal:

Plot the signal:

Let's compute both Energy and RMSE for sliding windows of 1024 samples. 

Notice that we do not have to choose a power of 2 anymore. However, when calculating multiple features (including spectral ones, where you *will* use power of 2) it is wise to examine the same slice (or collection) of samples.

All we have to do is implement the math into code:
$$ \sum_n \left| x(n) \right|^2 $$

In [ ]:
# establish a frame size and hop size:

hop_length = 512 # 50% overlap
frame_length = 1024

#### Energy = Total squared energy in each frame:

In [ ]:
energy = np.array([ # convert output from list comprehension into array
    
    # 1. grab one frame of samples at a time (i through i+frame_length)
    # 2. take the absolute value of the square of all the samples inside the frame, then sum
    # 3. increment by hop_length & repeat
    
])

#### RMS = average amplitude magnitude 

For now, we compute the RMS using [`librosa.feature.rms`](https://librosa.org/doc/latest/generated/librosa.feature.rms.html#librosa.feature.rms)

It has a handy function for converting the frame indices back to time values to align with the original input signal.

Plot both the energy and RMSE along with the waveform:

In [ ]:
librosa.display.waveshow(xnorm, sr=fs, alpha=0.4)
plt.plot(t, energy/energy.max(), 'r--', label='Energy')       # normalized for visualization
plt.plot(t, rmse[0]/rmse[0].max(), color='g', label='RMSE')   # normalized for visualization
plt.title('Normalized Signal with Energy and RMS overlaid')
plt.legend()

This plot lets us **see energy peaks, which often correspond to onsets.** (When there's an onset, the energy increases rapidly). Notice that what we have done with the RMSE is plot the envelope of the signal!

Keep in mind: We have normalized the energy and RMS values independently here to simply show the location of the respective values and "fit" them on top of the signal. However, **this scaling ignores their original differences in magnitude** so while it affords one kind of visualization, it is partially obstructing the true dynamics of each feature.

#### Why we tend to use RMS in onset detection rather than energy:

RMS provides a normalized measure of the signal's magnitude (i.e., it accounts for the average power). It’s often preferred in analysis because it gives you a smoother representation of the signal’s energy, making it easier to compare between frames.

It is less sensitive to noise because of this smoothness, which can give "false alarms" for onset detection. 

Because of the normalization, it's also easier to compare *across* different frames and tracks, which makes it more consistent. RMSE gives you a measure that is independent of the overall amplitude scale of the signal.

### Converting the Raw Signal to a dB Scale & log_energy

#### Log Energy (10 × log10)
Energy (power) in dB: We use 10 × log10 because *energy is proportional to the square of the amplitude*. To compute log energy (in decibels), you use the formula: 

$$ \text{log energy} = 10 \times \log_{10}(\text{energy})$$

The factor of 10 comes from the logarithmic relationship between energy and power (which is proportional to amplitude squared). This is why we use 10 × log10 for energy.

#### Log Amplitude (20 × log10)
To convert our original signal (currently normalized -1 to 1) into dB, you use the formula:

$$\text{Amplitude-in-dB}=20 \times \log_{10}(\text{amplitude})$$



Amplitude in dB: We use 20 × log10 because amplitude is the square root of energy, and logarithms of square roots introduce the factor of 2 (making it 20 instead of 10).



The log scale is commonly used in signal processing to handle large variations in signal amplitude (like energy) because it compresses high values and expands low ones.

In this case, we can compare more easily energy and RMSE if we log-scale the energy:

The log scaling of the energy helps compress the wide range of energy values into a more manageable scale, making it easier to compare with other signals like RMSE. 

Here's another visualization of the earlier plot but with our energy log-scaled:

In [ ]:

# convert energy to log energy (db)
log_energy = 10 * np.log10(energy + 1e-6)

# make subplots so you can have two different axes
# signal and energy on separate axes
fig, ax1 = plt.subplots()

time = np.arange(0,len(xnorm))/fs
ax1.plot(time, xnorm, alpha=0.5)
ax1.set_ylabel('Amplitude', color='lightblue')
ax2 = ax1.twinx()
ax2.plot(t, log_energy/max(abs(log_energy)), 'r--')
ax2.set_ylabel('Log Energy (dB)', color='r')


# Add labels and legend
plt.xlabel('Time (s)')
plt.ylabel('Log Energy (dB)')
plt.title('Signal and Log Energy on dB Scale')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example data:
# Assuming you have xnorm (signal), energy, rms, t (time), and fs (sampling rate)

# Convert energy to log energy (dB)
log_energy = 10 * np.log10(energy + 1e-6)  # Add small epsilon to avoid log(0)

# Make subplots so you can have two different axes
fig, ax1 = plt.subplots(figsize=(10, 6))

# Time vector for plotting (assuming 't' is your time vector for energy)
time = np.arange(0, len(xnorm)) / fs  # Time for signal plotting

# Plot the signal on the first axis (left)
ax1.plot(time, xnorm, alpha=0.5, label="Signal", color='lightblue')
ax1.set_ylabel('Amplitude', color='lightblue')

# Plot the RMS on the same left axis (ax1)
ax1.plot(t, rmse[0], color='g', label="RMSE")  # You can choose a color for RMS
ax1.set_ylabel('Amplitude / RMS', color='g')

# Create a second axis (right) for the log energy
ax2 = ax1.twinx()
ax2.plot(t, log_energy / max(abs(log_energy)), 'r--', label="Log Energy")
ax2.set_ylabel('Log Energy (dB)', color='r')

# Add labels and title
plt.xlabel('Time (s)')
plt.title('Signal, RMS, and Log Energy')

# Add legends for both axes
ax1.legend(loc='upper right')
ax2.legend(loc='upper right')

# Show the plot
plt.show()


### Novelty Function

Now that we have a way to calculate average magnitude of our signal, we can finish implementing our novelty function (onset detection function) and our peak picking function.

To complete our novelty function, we need to take the discrete derivative of our energy result. To do this, we just take the difference between two subsequent energy values. We will compute the derivative of the RMS envelope. This gives us the rate of change of energy, which highlights sudden increases.

After taking the difference between energy values, we also need to perform Half Wave Rectification (HWR). This just involves setting any negative values in our difference array to 0.

We do HWR because for onset detection, we only care about the time positions where energy increased. By setting negative diffs to 0, we ignore sections where energy decreased.

In [ ]:
novelty[novelty < 0] = 0

In [ ]:
plt.plot(t, novelty, label='Novelty Function')
plt.xlabel('Time (s)')
plt.title('Novelty Function from RMSE')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(t, rmse[0]/rmse[0].max(), label='Normalized RMS')
plt.plot(t, novelty/novelty.max(), label='Normalized Novelty')
plt.legend()
plt.title('RMS vs Novelty (Derivative)')
plt.show()

Now that we have our novelty function with clearer onsets, we need to decide a peak picking function.

For now, we can just use an easy threshold where onsets exist where our novelty function is greater than a certain value. We will look for local maxima above .1 in our case.

In [ ]:
plt.plot(t, novelty, label='Novelty Function')
plt.vlines(peak_times, 0, np.max(novelty), colors='r')
plt.xlabel('Time (s)')
plt.title('Novelty Function with Threshold Peaks')
plt.legend()
plt.show()

We can make this more complex and (likely) more accurate. For example, even with normalization the scale and shape of our audio signals can change across data inputs so a fixed threshold may not work. We can instead apply an adaptive threshold with filtering.

Additionally, we may want to considering timing (spacing) of onsets in our detection function. We also may want to consider tightly spaced onsets with large amplitude distances.

Let's take our manual peak picking one step further by applying thresholding to local maxima only.

In [ ]:
threshold = 0.1
peak_indices = []

for i in range(1, len(novelty)-1):
    if novelty[i] > threshold:
        if novelty[i] > novelty[i-1] and novelty[i] > novelty[i+1]:
            peak_indices.append(i)

peak_indices = np.array(peak_indices)
peak_times = t[peak_indices]

In [ ]:
plt.plot(t, novelty, label='Novelty Function')
plt.vlines(peak_times, 0, np.max(novelty), colors='r')
plt.xlabel('Time (s)')
plt.title('Novelty Function with Threshold Peaks')
plt.legend()
plt.show()

Because this is a drum sound and we had clear peaks from our onset detection function, this did not give a huge change.

### Compare to built in functions

Librosa has a built-in onset detection function.

Librosa's onset detection function uses spectral flux in the calculation of the novelty function (called onset envelope).

In [ ]:
x, fs = librosa.load('../audio/CongaGroove-mono.wav')
hop_length = 512 # 50% overlap
frame_length = 1024

# Get the frame->beat strength profile
onset_envelope = librosa.onset.onset_strength(y=x, sr=fs,hop_length=hop_length)
# Locate note onset events
onsets = librosa.onset.onset_detect(y=x,sr=fs,onset_envelope=onset_envelope,hop_length=hop_length)
# Convert frames to time
times = librosa.frames_to_time(np.arange(len(onset_envelope)), sr=fs, hop_length=hop_length)

print('Peaks', onsets)
print('Times', times[onsets])

In [ ]:
plt.plot(times, onset_envelope, alpha=0.8, label='Novelty Function')
plt.vlines(times[onsets], 0, np.max(onset_envelope), color='r')
plt.legend

Scipy also has a built-in peak picking function called find_peaks.

We can use this on our novelty function.

In the function, we can set parameters like height for thresholding and minimum distance between peaks. For adaptive thresholding, we still need to pass a filter/envelope to account for a moving threshold ourrselves.

In [ ]:
peaks, properties = scipy.signal.find_peaks(novelty, height=0.1)

peak_times = t[peaks]

plt.plot(t, novelty)
plt.vlines(peak_times, 0, np.max(novelty), colors='r')
plt.title('find_peaks Result')
plt.show()

### Onset detection with Spectrum

An alternative approach uses spectral flux. Recall that spectral flux attempts to capture fast changing properties of the spectral content of the signal. Thus transients are well-captured with spectral flux.

* Compute STFT (optional: convert to log scale)
* Calculate spectral flux*
* Use thresholding procedure same as above.

Notes:

* less efficient & more complicated calculations (must sum over all frequency bins & use FFT)
* in some cases you may intentionally want to *only* sum over certain frequency bands (e.g., noise-like transient power will show up well above 1-2kHz)
* works slightly better for less percussive tracks with soft transients (e.g., legato notes, woodwinds, etc.)

### Tempo Estimation

How to estimate the tempo of a clip of music? We assume that onsets occur most frequently on beats, and beats are articulated more strongly and more frequently than off-beats, and should be approximately evenly spaced in time. (We also assume that the tempo is not changing for the purpose of this class!!)

Given this assumption, we need to examine the regularity or *periodicity* of the onsets. More on that next!